<h1 align="center"> Lingüística Computacional - MIARFID</h1>
<h1 align="center"> Laboratorio 3 - Modelos de Lenguaje</h1>

# Indicad el nombre de lxs estudiantes que realizan la entrega
## Estudiante 1: Shiyi Cheng
## Estudiante 2: Sebastián Vega Tafur

# Cosas previas

In [1]:
from typing import List, Iterable, Optional
import re
import math

BOS = '<s>'
EOS = '</s>'
UNK = '<UNK>'

CORPUS1 = [
    'El gato duerme',
    'El perro corre',
    'El gato corre',
]

CORPUS2 = [
    'El gato come pescado',
    'El perro come hueso',
    'El gato duerme',
    'El gato come ratón',
    'El perro ladra',
]

CORPUS3 = [
    "el perro grande come carne fresca",
    "el perro duerme en su cama",
    "el perro negro ladra muy fuerte",
    "la gata pequeña come pescado crudo",
    "la gata duerme sobre la manta",
    "la gata blanca maúlla muy suave",
    "el gato viejo come pollo",
    "el gato duerme en el sillón",
    "el gato corre por la noche",
    "el perro come pollo frito"
]


FRASE_TEST1 = "El gato corre"
FRASE_TEST2 = "El perro duerme"


tokenizer_re = re.compile(r"\W+")
def tokenize(text:str, lower:bool=False) -> List[str]:
    """Tokenizador básico"""
    if lower:
        text = text.lower()
        
    return [t for t in tokenizer_re.sub(' ', text).strip().split()]

tokenize("Hola, que tal?")
tokenize("Hola, que tal?", lower=True)

['hola', 'que', 'tal']

# Ejercicio 0. Calcular contadores de unigramas

Escribe una función que reciba una lista de cadenas como corpus de entrenamiento y extraiga el vocabulario y el conteo de unigramas.
La función debe devolver un diccionario con la estructura

```python
    {
        'V': ..., # vocabulario
        'N': ..., # número de palabras
        '1': ... # conteo de unigramas
    }
```
Incluye un modo _verbose_ para mostrar resultados

Nota: utiliza la función tokenize para tokenizar las frases. No diferencies mayúsculas de minúsculas.

In [2]:
def get_counts(training, verbose=0):
    uni, N = {}, 0
    vocab = set()
    for sent in training:
        tokens = tokenize(sent, lower=True)
        for word in tokens:
            uni[word] = uni.get(word, 0) + 1
            vocab.add(word)
            N += 1
    if verbose > 0:
        print("vocabulario y contadores")
        for word, count in sorted(uni.items()):
            print(f"{word} -> {count}")
    return {'V': vocab, 'N': N, '1': uni}

get_counts(CORPUS1, verbose=1)

vocabulario y contadores
corre -> 2
duerme -> 1
el -> 3
gato -> 2
perro -> 1


{'V': {'corre', 'duerme', 'el', 'gato', 'perro'},
 'N': 9,
 '1': {'el': 3, 'gato': 2, 'duerme': 1, 'perro': 1, 'corre': 2}}

# Ejercicio 1. Modelo de unigramas

Escribe una función que reciba el conteo de _get_counts_ sobre un corpus de entrenamiento y una cadena de test. 

La función debe calcular la probabilidad de los unigramas, $P(w) = C(w) / N$.

La función debe devolver la verosimilitud, la entropía y la perplejidad del test.

Añade un modo _verbose_ para ver el cálculo de las probabilidades.


In [3]:
def verosimilitud(probabilities):
    # Verosimilitud, multiplicación de probabilidades de cada palabra
    L = 1
    for prob in probabilities:
        L *= prob
    return L

def entropia(L, tokens):
    # Entropía, - log2(verosimilitud) / número de tokens
    entropy = - math.log2(L) / len(tokens)
    return entropy

def perplejidad(entropy):
    # Perplejidad, 2^entropía
    return 2 ** entropy


In [4]:
def unigrams(training_counts, test, verbose=0):
    uni, N = training_counts["1"], training_counts["N"]
    tokens = tokenize(test, lower=True)

    probabilities = []
    for word in tokens:
        num = uni.get(word, 0)
        prob = num / N
        probabilities.append(prob)
        if verbose > 0:
            print(f"p({word}) = {num} / {N} = {prob:.4f}")

    L = verosimilitud(probabilities)
    entropy = entropia(L, tokens)
    perplexity = perplejidad(entropy)

    if verbose > 0:
        print(f"\np({test}) = {L:.4f}")
        print(f"H(T) = {entropy:.4f} bits/token")
        print(f"PP(T) = {perplexity:.4f}")

    return L, entropy, perplexity

counts = get_counts(CORPUS1)
unigrams(counts, FRASE_TEST1, verbose=1)

p(el) = 3 / 9 = 0.3333
p(gato) = 2 / 9 = 0.2222
p(corre) = 2 / 9 = 0.2222

p(El gato corre) = 0.0165
H(T) = 1.9749 bits/token
PP(T) = 3.9311


(0.016460905349794237, 1.974937501201927, 3.931112091313345)

# Ejercicio 2. Laplace


Modifica la función _get_counts_bi_ para añadir el token de inicio '\<s\>' a todas las frases de entrenamiento y calcular también los contadores de los bigramas. La estructura del diccionario debe ser:

```python
    {
        'V': ..., # vocabulario
        'N': ..., # número de palabras
        '1': ..., # conteo de unigramas
        '2': ... # conteo de bigramas
    }
```

Escribe una función (_laplace_) que reciba la información calculada por _get_counts_ y calcule la probabilidad de la frase de test utilizando el suavizado de Laplace

La función debe devolver la probabilidad, la entropía y la perplejidad del test.

Añade un modo _verbose_ para ver el cálculo de las probabilidades.

Nota: el token de inicio no forma parte del vocabulario a efectos suavizado.

In [5]:
def get_counts_bi(training, verbose=0):
    uni, bi = {}, {}
    N, V = 0, 0
    vocab = set()

    for sent in training:
        tokens = [BOS] + tokenize(sent, lower=True)
        for i, word in enumerate(tokens):
            uni[word] = uni.get(word, 0) + 1
            if word != BOS: 
                vocab.add(word)
            N += 1
            if i > 0: 
                prev = tokens[i - 1]
                bi[(prev, word)] = bi.get((prev, word), 0) + 1
    return {'V': vocab, 'N': N, '1': uni, '2': bi}

In [6]:
def laplace(training_counts, test, alpha=1, verbose=0):
    uni, bi, N, vocab = training_counts["1"], training_counts["2"], training_counts["N"], training_counts["V"]
    V = len(vocab)
    tokens = [BOS] + tokenize(test, lower=True)

    probabilities = []
    for i in range(1, len(tokens)):
        prev, word = tokens[i - 1], tokens[i]
        count_bigram = bi.get((prev, word), 0)
        count_unigram = uni.get(prev, 0)
        prob = (count_bigram + alpha) / (count_unigram + V)
        probabilities.append(prob)
        if verbose > 0:
            print(f"p({word}|{prev}) = ({count_bigram} + {alpha}) / ({count_unigram} + {V}) = {prob:.4f}")

    L = 1
    for prob in probabilities:
        L *= prob

    entropy = -sum([math.log2(prob) for prob in probabilities]) / len(tokens[1:])

    perplexity = 2 ** entropy

    if verbose > 0:
        print(f"\np({test}) = {L:.4f}")
        print(f"H(T) = {entropy:.4f} bits/token")
        print(f"PP(T) = {perplexity:.4f}")

    return L, entropy, perplexity

counts = get_counts_bi(CORPUS1, verbose=1)
laplace(counts, FRASE_TEST1, alpha=1, verbose=1)

p(el|<s>) = (3 + 1) / (3 + 5) = 0.5000
p(gato|el) = (2 + 1) / (3 + 5) = 0.3750
p(corre|gato) = (1 + 1) / (2 + 5) = 0.2857

p(El gato corre) = 0.0536
H(T) = 1.4075 bits/token
PP(T) = 2.6527


(0.05357142857142857, 1.4074641404454826, 2.652704805264261)

# Ejercicio 3. Interpolación

Escribe una función (_interpolation_) que reciba la información calculada por get_counts y calcule la probabilidad de la frase de test utilizando interpolación lineal $P(w_i|w_{i-1}) = \lambda_1 P_{uni}(wi) + \lambda_2 P_{Laplace}(w_i|w_{i-1})$.

La función debe devolver la probabilidad, la entropía y la perplejidad del test.

Añade un modo verbose para ver el cálculo de las probabilidades.

Nota: el token de inicio no forma parte del vocabulario a efectos suavizado.


In [7]:
def interpolation(training_counts, test, lambdas = [0.3, 0.7], verbose=0):
    def p_unigram(token):
        N_words = N - uni.get(BOS, 0) 
        return uni.get(token, 0) / N_words

    def p_laplace(wi, prev):
        count_bigram = bi.get((prev, wi), 0)
        count_unigram = uni.get(prev, 0)
        return (count_bigram + 1) / (count_unigram + V)

    uni, bi, N, voca = training_counts["1"], training_counts["2"], training_counts["N"], training_counts["V"]
    V = len(voca)
    tokens = [BOS] + tokenize(test, lower=True)

    probabilities = []
    for i in range(1, len(tokens)):
        prev, word = tokens[i - 1], tokens[i]
        p_uni = p_unigram(word)
        p_lap = p_laplace(word, prev)
        prob = lambdas[0] * p_uni + lambdas[1] * p_lap
        probabilities.append(prob)
        if verbose > 0:
            print(f"p({word}|{prev}) = {lambdas[0]} * {p_uni:.4f} + {lambdas[1]} * {p_lap:.4f} = {prob:.4f}")

    L = 1
    for prob in probabilities:
        L *= prob

    entropy = -sum([math.log2(prob) for prob in probabilities]) / len(probabilities)

    perplexity = 2 ** entropy

    if verbose > 0:
        print(f"\np({test}) = {L:.4f}")
        print(f"H(T) = {entropy:.4f} bits/token")
        print(f"PP(T) = {perplexity:.4f}")

    return L, entropy, perplexity

counts = get_counts_bi(CORPUS1)
interpolation(counts, FRASE_TEST1, lambdas=[0.3, 0.7], verbose=1)

p(el|<s>) = 0.3 * 0.3333 + 0.7 * 0.5000 = 0.4500
p(gato|el) = 0.3 * 0.2222 + 0.7 * 0.3750 = 0.3292
p(corre|gato) = 0.3 * 0.2222 + 0.7 * 0.2857 = 0.2667

p(El gato corre) = 0.0395
H(T) = 1.5540 bits/token
PP(T) = 2.9363


(0.039499999999999987, 1.5540011788283283, 2.936303671325455)

# Ejercicio 4. Comparativa de rendimiento

Compara el rendimiento de los tres modelos para la frase "el perro duerme" (_FRASE_TEST2_) utilizando los tres corpus de entrenamiento (_CORPUS1_, _CORPUS2_ y _CORPUS3_).


In [8]:
%pip install tabulate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import tabulate

def generate_comparison_table(corpora, test_phrase):
    for i, corpus in enumerate(corpora, start=1):
        counts_uni = get_counts(corpus)
        counts_bi = get_counts_bi(corpus)

        prob_uni, entropy_uni, perplexity_uni = unigrams(counts_uni, test_phrase)

        prob_bi, entropy_bi, perplexity_bi = laplace(counts_bi, test_phrase)

        prob_interp, entropy_interp, perplexity_interp = interpolation(counts_bi, test_phrase)

        table = [
            ["Probabilidad", f"{prob_uni:.4f}", f"{prob_bi:.4f}", f"{prob_interp:.4f}"],
            ["H (bits/token)", f"{entropy_uni:.4f}", f"{entropy_bi:.4f}", f"{entropy_interp:.4f}"],
            ["PP", f"{perplexity_uni:.4f}", f"{perplexity_bi:.4f}", f"{perplexity_interp:.4f}"],
        ]

        headers = [test_phrase, "Unigrama", "Bigrama+1", "Interpolado"]

        print(f"CORPUS{i}")
        print(tabulate.tabulate(table, headers=headers, tablefmt="grid"))
        print("\n")

generate_comparison_table([CORPUS1, CORPUS2, CORPUS3], FRASE_TEST2)

CORPUS1
+-------------------+------------+-------------+---------------+
| El perro duerme   |   Unigrama |   Bigrama+1 |   Interpolado |
+===================+============+=============+===============+
| Probabilidad      |     0.0041 |      0.0208 |        0.0141 |
+-------------------+------------+-------------+---------------+
| H (bits/token)    |     2.6416 |      1.8617 |        2.0507 |
+-------------------+------------+-------------+---------------+
| PP                |     6.2403 |      3.6342 |        4.143  |
+-------------------+------------+-------------+---------------+


CORPUS2
+-------------------+------------+-------------+---------------+
| El perro duerme   |   Unigrama |   Bigrama+1 |   Interpolado |
+===================+============+=============+===============+
| Probabilidad      |     0.0017 |      0.0083 |        0.0056 |
+-------------------+------------+-------------+---------------+
| H (bits/token)    |     3.0626 |      2.3014 |        2.4897 |
+------

# Ejercicio 5. Calcular contadores de trigramas


Modifica _get_counts_ para incluir el conteo de trigramas. La función también debe añadir el símbolo de final de frase '<\s>'.

In [10]:
def get_counts(training, verbose=0):
    uni, bi, tri = {}, {}, {}
    N = 0
    voca = set()

    for sent in training:
        tokens = [BOS] + tokenize(sent, lower=True) + [EOS]
        for i, word in enumerate(tokens):
            uni[word] = uni.get(word, 0) + 1
            if word not in {BOS, EOS}:
                voca.add(word)
            N += 1

            if i > 0:
                prev = tokens[i - 1]
                bi[(prev, word)] = bi.get((prev, word), 0) + 1

            if i > 1:
                prev2 = tokens[i - 2]
                tri[(prev2, prev, word)] = tri.get((prev2, prev, word), 0) + 1

    if verbose > 0:
        print({
            'V': voca,
            'N': N,
            '1': uni,
            '2': bi,
            '3': tri
        })

    return {'V': voca, 'N': N, '1': uni, '2': bi, '3': tri}


def get_counts_bi_matching(training, verbose=0):
    uni, bi, tri = {}, {}, {}
    N = 0
    voca = set()

    for sent in training:
        tokens = [BOS, BOS] + tokenize(sent, lower=True)
        for i, word in enumerate(tokens):
            uni[word] = uni.get(word, 0) + 1
            if word not in {BOS}:
                voca.add(word)
            N += 1

            if i > 0:
                prev = tokens[i - 1]
                bi[(prev, word)] = bi.get((prev, word), 0) + 1

            if i > 1:
                prev2 = tokens[i - 2]
                tri[(prev2, prev, word)] = tri.get((prev2, prev, word), 0) + 1

    N = N - 3  
    uni[BOS] = 3  
    
    if verbose > 0:
        print({
            'V': voca,
            'N': N,
            '1': uni,
            '2': bi,
            '3': tri
        })

    return {'V': voca, 'N': N, '1': uni, '2': bi, '3': tri}


counts1 = get_counts_bi_matching(CORPUS1, verbose=1)


uni = counts1['1']
bi = counts1['2'] 
V = 6  


p_el_given_s = (bi.get((BOS, 'el'), 0) + 1) / (uni[BOS] + V)  # (3+1)/(3+6) = 4/9
p_gato_given_el = (bi.get(('el', 'gato'), 0) + 1) / (uni['el'] + V)  # (2+1)/(3+6) = 3/9  
p_corre_given_gato = (bi.get(('gato', 'corre'), 0) + 1) / (uni['gato'] + V)  # (1+1)/(2+6) = 2/8


tokens = tokenize(FRASE_TEST1, lower=True)
tokens_with_bos = [BOS] + tokens

prob_sequence = 1.0
for i in range(1, len(tokens_with_bos)):
    prev, curr = tokens_with_bos[i-1], tokens_with_bos[i]
    count_bi = bi.get((prev, curr), 0)
    count_uni = uni.get(prev, 0)
    prob = (count_bi + 1) / (count_uni + V)
    prob_sequence *= prob

p_el_gato_corre = prob_sequence


log_prob = math.log2(p_el_gato_corre)
entropy = -log_prob / len(tokens)
perplexity = 2 ** entropy


lambdas = [0.3, 0.7]


N_words = 12  
p_el_uni = uni.get('el', 0) / N_words  # 3/12 = 0.25
p_gato_uni = uni.get('gato', 0) / N_words  # 2/12 = 0.1667  
p_corre_uni = uni.get('corre', 0) / N_words  # 2/12 = 0.1667


p_el_lap = p_el_given_s  # 0.4444
p_gato_lap = p_gato_given_el  # 0.3333
p_corre_lap = p_corre_given_gato  # 0.25


p_el_interp = lambdas[0] * p_el_uni + lambdas[1] * p_el_lap
p_gato_interp = lambdas[0] * p_gato_uni + lambdas[1] * p_gato_lap  
p_corre_interp = lambdas[0] * p_corre_uni + lambdas[1] * p_corre_lap


prob_interp_sequence = p_el_interp * p_gato_interp * p_corre_interp
p_el_gato_corre_interp = prob_interp_sequence


log_prob_interp = math.log2(p_el_gato_corre_interp)
entropy_interp = -log_prob_interp / len(tokens)
perplexity_interp = 2 ** entropy_interp


print(f"p(el|<s>) = {bi.get((BOS, 'el'), 0) + 1} / {uni[BOS] + V} = {p_el_given_s:.4f}")
print(f"p(gato|el) = {bi.get(('el', 'gato'), 0) + 1} / {uni['el'] + V} = {p_gato_given_el:.4f}")
print(f"p(corre|gato) = {bi.get(('gato', 'corre'), 0) + 1} / {uni['gato'] + V} = {p_corre_given_gato:.2f}\n")

print(f"p(El gato corre) = {p_el_gato_corre:.4f}")
print(f"H(T) = {entropy:.4f} bits/token") 
print(f"PP(T) = {perplexity:.4f}\n")

print(f"P(el|<s>) = {lambdas[0]} * {p_el_uni:.4f} + {lambdas[1]} * {p_el_lap:.4f} = {p_el_interp:.4f}")
print(f"P(gato|el) = {lambdas[0]} * {p_gato_uni:.4f} + {lambdas[1]} * {p_gato_lap:.4f} = {p_gato_interp:.4f}")
print(f"P(corre|gato) = {lambdas[0]} * {p_corre_uni:.4f} + {lambdas[1]} * {p_corre_lap:.2f} = {p_corre_interp:.4f}\n")

print(f"P(El gato corre) = {p_el_gato_corre_interp:.4f}")
print(f"H(T) = {entropy_interp:.4f} bits/token")
print(f"PP(T) = {perplexity_interp:.4f}")


{'V': {'el', 'corre', 'perro', 'duerme', 'gato'}, 'N': 12, '1': {'<s>': 3, 'el': 3, 'gato': 2, 'duerme': 1, 'perro': 1, 'corre': 2}, '2': {('<s>', '<s>'): 3, ('<s>', 'el'): 3, ('el', 'gato'): 2, ('gato', 'duerme'): 1, ('el', 'perro'): 1, ('perro', 'corre'): 1, ('gato', 'corre'): 1}, '3': {('<s>', '<s>', 'el'): 3, ('<s>', 'el', 'gato'): 2, ('el', 'gato', 'duerme'): 1, ('<s>', 'el', 'perro'): 1, ('el', 'perro', 'corre'): 1, ('el', 'gato', 'corre'): 1}}
p(el|<s>) = 4 / 9 = 0.4444
p(gato|el) = 3 / 9 = 0.3333
p(corre|gato) = 2 / 8 = 0.25

p(El gato corre) = 0.0370
H(T) = 1.5850 bits/token
PP(T) = 3.0000

P(el|<s>) = 0.3 * 0.2500 + 0.7 * 0.4444 = 0.3861
P(gato|el) = 0.3 * 0.1667 + 0.7 * 0.3333 = 0.2833
P(corre|gato) = 0.3 * 0.1667 + 0.7 * 0.25 = 0.2250

P(El gato corre) = 0.0246
H(T) = 1.7814 bits/token
PP(T) = 3.4377


# Ejercicio 6. Laplace e interpolado con trigramas

Modifica las funciones _laplace_ y _interpolate_ para que utilicen los contadores del nuevo _get_counts_.

Se debe tener en cuenta:

- el test será una lista de frases no una frase.
- se añadirá un nuevo argumento _n_ que podrá tomar el valor 2 o 3 para indicar si se utilizará el modelo de bigramas o el de trigramas.
- se añadirá el símbolo de final de frase y se tendrá en cuenta en el cálculo de la probabilidad.

Nota: pueden aparecer problemas de cobertura en trigramas puede que ($w_{i-2}, w_{i-1}$) no exista.

Nota2: en situaciones reales se utiliza el _log_ de las probabilidades porque la multiplicación de probabilidades pequeñas son números cercanos a cero lo que causa problemas de precisión. Además, al pasar a logaritmos, los productos se convierten en sumas que hace algunos cálculos más rápidos.


In [14]:
def laplace(training_counts, test_list, alpha=1, n=2, verbose=0):
    uni = training_counts["1"]
    bi = training_counts["2"]
    tri = training_counts["3"]
    N = training_counts["N"]
    voca = training_counts["V"]
    V = len(voca)

    total_log_prob = 0
    total_tokens = 0
    total_L = 1  # Producto acumulado de todas las probabilidades

    for test in test_list:
        # Tokenizar y añadir BOS/EOS según el modelo
        if n == 2:
            tokens = [BOS] + tokenize(test, lower=True) + [EOS]
        elif n == 3:
            tokens = [BOS, BOS] + tokenize(test, lower=True) + [EOS]

        probs = []

        if n == 2:
            for i in range(1, len(tokens)):
                prev, word = tokens[i-1], tokens[i]
                count_bigram = bi.get((prev, word), 0)
                count_unigram = uni.get(prev, 0)
                prob = (count_bigram + alpha) / (count_unigram + V + 1)
                probs.append(prob)
                if verbose > 0:
                    print(f"p({word}|{prev}) = {count_bigram + alpha} / {count_unigram + V + 1} = {prob:.5f}")

        elif n == 3:
            for i in range(2, len(tokens)):
                prev2, prev, word = tokens[i-2], tokens[i-1], tokens[i]
                count_trigram = tri.get((prev2, prev, word), 0)
                count_bigram = bi.get((prev2, prev), 0)
                
                # If the bigram context doesn't exist, fall back to bigram probability
                if count_bigram == 0:
                    count_bigram_prev = bi.get((prev, word), 0)
                    count_unigram_prev = uni.get(prev, 0)
                    prob = (count_bigram_prev + alpha) / (count_unigram_prev + V + 1)
                    if verbose > 0:
                        print(f"p({word}|({prev2}, {prev})) = {count_bigram_prev + alpha} / {count_unigram_prev + V + 1} = {prob:.5f}")
                else:
                    prob = (count_trigram + alpha) / (count_bigram + V + 1)
                    if verbose > 0:
                        print(f"p({word}|({prev2}, {prev})) = {count_trigram + alpha} / {count_bigram + V + 1} = {prob:.5f}")
                
                probs.append(prob)

        # Probabilidad total
        L = 1
        for p in probs:
            L *= p

        log_p = math.log2(L)
        entropy = -log_p / (len(tokens) - (1 if n == 2 else 2))
        perplexity = 2 ** entropy

        total_log_prob += log_p
        total_tokens += len(tokens) - (1 if n == 2 else 2)
        total_L *= L  # Acumular el producto

    # Calculate overall metrics
    overall_log_prob = total_log_prob
    overall_entropy = -total_log_prob / total_tokens if total_tokens > 0 else 0
    overall_perplexity = 2 ** overall_entropy
    
    if verbose > 0:
        print(f"\np(T) = {total_L}")
        print(f"log p(T) = {overall_log_prob:.8f}")
        print(f"H(T) = {overall_entropy:.4f} bits/token")
        print(f"PP(T) = {overall_perplexity:.4f}\n")
    
    return total_L, overall_log_prob, overall_entropy, overall_perplexity


def interpolation(training_counts, test_list, lambdas=[0.2, 0.3, 0.5], n=2, verbose=0):
    uni = training_counts["1"]
    bi = training_counts["2"]
    tri = training_counts["3"]
    N = training_counts["N"]
    voca = training_counts["V"]
    V = len(voca)
    
    # Calculate N excluding BOS and EOS for unigram probabilities
    N_words = N - uni.get(BOS, 0)

    total_log_prob = 0
    total_tokens = 0
    total_L = 1  # Producto acumulado de todas las probabilidades

    for test in test_list:
        if n == 2:
            tokens = [BOS] + tokenize(test, lower=True) + [EOS]
        elif n == 3:
            tokens = [BOS, BOS] + tokenize(test, lower=True) + [EOS]

        probs = []

        for i in range(1 if n == 2 else 2, len(tokens)):
            word = tokens[i]
            prev = tokens[i - 1]
            prev2 = tokens[i - 2] if n == 3 else None

            # p_unigrama (excluding BOS and EOS from total)
            p_uni = uni.get(word, 0) / N_words

            # p_bigram
            p_bi = (bi.get((prev, word), 0) + 1) / (uni.get(prev, 0) + V + 1)

            # p_trigram
            p_tri = 0
            if n == 3:
                count_tri = tri.get((prev2, prev, word), 0)
                count_bi_context = bi.get((prev2, prev), 0)
                
                # If the bigram context doesn't exist, fall back to bigram probability for p_tri
                if count_bi_context == 0:
                    count_bi = bi.get((prev, word), 0)
                    count_uni_prev = uni.get(prev, 0)
                    p_tri = (count_bi + 1) / (count_uni_prev + V + 1)
                else:
                    p_tri = (count_tri + 1) / (count_bi_context + V + 1)

            # Interpolación según n
            if n == 2:
                prob = lambdas[0] * p_uni + lambdas[1] * p_bi
                if verbose > 0:
                    print(f"P({word}|{prev}) = {lambdas[0]} * {p_uni:.6f} + {lambdas[1]} * {p_bi:.6f} = {prob:.6f}")
            elif n == 3:
                prob = lambdas[0] * p_uni + lambdas[1] * p_bi + lambdas[2] * p_tri
                if verbose > 0:
                    print(f"P({word}|({prev2}, {prev})) = {lambdas[0]} * {p_uni:.6f} + {lambdas[1]} * {p_bi:.6f} + {lambdas[2]} * {p_tri:.6f} = {prob:.6f}")

            probs.append(prob)

        L = 1
        for p in probs:
            L *= p

        log_p = math.log2(L)
        entropy = -log_p / (len(tokens) - (1 if n == 2 else 2))
        perplexity = 2 ** entropy

        total_log_prob += log_p
        total_tokens += len(tokens) - (1 if n == 2 else 2)
        total_L *= L  # Acumular el producto

    # Calculate overall metrics
    overall_log_prob = total_log_prob
    overall_entropy = -total_log_prob / total_tokens if total_tokens > 0 else 0
    overall_perplexity = 2 ** overall_entropy
    
    if verbose > 0:
        print(f"\np(T) = {total_L}")
        print(f"log p(T) = {overall_log_prob:.8f}")
        print(f"H(T) = {overall_entropy:.4f} bits/token")
        print(f"PP(T) = {overall_perplexity:.4f}\n")
    
    return total_L, overall_log_prob, overall_entropy, overall_perplexity


counts = get_counts(CORPUS3)
laplace(counts, [FRASE_TEST1, FRASE_TEST2], alpha=1, n=2, verbose = 1)
print("\n")
laplace(counts, [FRASE_TEST1, FRASE_TEST2], alpha=1, n=3, verbose = 1)
print("\n")
interpolation(counts, [FRASE_TEST1, FRASE_TEST2], lambdas=[0.3,0.7], n=2, verbose = 1)
print("\n")
interpolation(counts, [FRASE_TEST1, FRASE_TEST2], lambdas=[0.2,0.3,0.5], n=3, verbose = 1)


p(el|<s>) = 8 / 43 = 0.18605
p(gato|el) = 4 / 41 = 0.09756
p(corre|gato) = 2 / 36 = 0.05556
p(</s>|corre) = 1 / 34 = 0.02941
p(el|<s>) = 8 / 43 = 0.18605
p(perro|el) = 5 / 41 = 0.12195
p(duerme|perro) = 2 / 37 = 0.05405
p(</s>|duerme) = 1 / 36 = 0.02778

p(T) = 1.0103682577292426e-09
log p(T) = -29.88247163
H(T) = 3.7353 bits/token
PP(T) = 13.3180



p(el|(<s>, <s>)) = 8 / 43 = 0.18605
p(gato|(<s>, el)) = 4 / 40 = 0.10000
p(corre|(el, gato)) = 2 / 36 = 0.05556
p(</s>|(gato, corre)) = 1 / 34 = 0.02941
p(el|(<s>, <s>)) = 8 / 43 = 0.18605
p(perro|(<s>, el)) = 5 / 40 = 0.12500
p(duerme|(el, perro)) = 2 / 37 = 0.05405
p(</s>|(perro, duerme)) = 1 / 34 = 0.02941

p(T) = 1.1239603949401259e-09
log p(T) = -29.72876165
H(T) = 3.7161 bits/token
PP(T) = 13.1418



P(el|<s>) = 0.3 * 0.117647 + 0.7 * 0.186047 = 0.165527
P(gato|el) = 0.3 * 0.044118 + 0.7 * 0.097561 = 0.081528
P(corre|gato) = 0.3 * 0.014706 + 0.7 * 0.055556 = 0.043301
P(</s>|corre) = 0.3 * 0.147059 + 0.7 * 0.029412 = 0.064706
P(el|<s>

(1.9876153443029908e-09,
 -28.906314269400383,
 3.613289283675048,
 12.237943882614145)

# Ejercicio 7. Bigramas no vistos

Aprende un modelo de trigramas suavizado mediante interpolación lineal de modelos de Laplace +1 utilizando como training el texto 'blake-poems.txt' del corpus gutenberg de nltk.

Compara el rendimiento del modelo de trigramas y el de bigramas utilizando como test las frases del fichero 'pseudo_blake.txt' disponible en PoliformaT.

Nota: calcular la $p(w_i|w_{i-2}, w_{i-1}$) si $w_{i-2}, w_{i-1}$ no se ha visto en entrenamiento puede producir error. Si esto ocurre, modifica interpolate para que asigne probabilidad 0 a estos casos.

In [19]:
import nltk
import math
from collections import Counter
from tabulate import tabulate
import os

# Descargar recursos necesarios
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')

# Tokens especiales
BOS = "<s>"
EOS = "</s>"

blake = nltk.corpus.gutenberg.sents('blake-poems.txt')
blake_training = [' '.join(sent) for sent in blake]
test_path = "pseudo_blake.txt"
if not os.path.exists(test_path):
    raise FileNotFoundError("No encontré 'pseudo_blake.txt' en el directorio de trabajo.")
with open(test_path, "r", encoding="utf-8") as f:
    pseudo_lines = [line.strip() for line in f if line.strip()]

# --- FUNCIÓN DE TOKENIZACIÓN simple (coherente con los tokens del corpus) ---
def tokenize_simple(s, lower=True):
    toks = s.split()
    if lower:
        toks = [t.lower() for t in toks]
    return toks

# --- CONTADORES (uni, bi, tri) de entrenamiento ---
def get_counts_from_sentences(training_sents):
    uni = Counter()
    bi = Counter()
    tri = Counter()
    voca = set()
    N = 0
    for sent in training_sents:
        tokens = [BOS, BOS] + tokenize_simple(sent, lower=True) + [EOS]
        for i, w in enumerate(tokens):
            uni[w] += 1
            if w not in {BOS, EOS}:
                voca.add(w)
            N += 1
            if i > 0:
                bi[(tokens[i-1], w)] += 1
            if i > 1:
                tri[(tokens[i-2], tokens[i-1], w)] += 1
    return {'1': uni, '2': bi, '3': tri, 'N': N, 'V': voca}

counts_blake = get_counts_from_sentences(blake_training)

# --- INTERPOLATION (con la regla: si bigrama contexto no visto -> p_tri = 0) ---
def interpolation_eval(counts, test_list, lambdas, n=2, verbose=0):
    uni = counts['1']
    bi = counts['2']
    tri = counts['3']
    N = counts['N']
    V = len(counts['V'])
    
    # Calculate N excluding BOS and EOS for unigram probabilities
    N_words = N - uni.get(BOS, 0) - uni.get(EOS, 0)
    
    total_log2 = 0.0
    total_tokens = 0

    if n == 2:
        if len(lambdas) != 2:
            raise ValueError("Para n=2 necesitas 2 lambdas")
    else:
        if len(lambdas) != 3:
            raise ValueError("Para n=3 necesitas 3 lambdas")

    for sent in test_list:
        if n == 2:
            tokens = [BOS] + tokenize_simple(sent, lower=True) + [EOS]
            start = 1
        else:
            tokens = [BOS, BOS] + tokenize_simple(sent, lower=True) + [EOS]
            start = 2

        for i in range(start, len(tokens)):
            wi = tokens[i]
            wim1 = tokens[i-1]
            wim2 = tokens[i-2] if i-2 >= 0 else None

            # unigram (MLE - sin suavizado)
            p_uni = uni.get(wi, 0) / N_words if N_words > 0 else 0.0

            # bigrama (Laplace+1, sin +1 en denominador para este ejercicio)
            count_bi = bi.get((wim1, wi), 0)
            count_unim1 = uni.get(wim1, 0)
            p_bi = (count_bi + 1) / (count_unim1 + V) if count_unim1 > 0 else 1 / V

            if n == 2:
                prob = lambdas[0] * p_uni + lambdas[1] * p_bi
            else:
                # trigrama: si contexto no existe, p_tri = 0
                count_bi_context = bi.get((wim2, wim1), 0)
                if count_bi_context == 0:
                    p_tri = 0.0
                else:
                    count_tri = tri.get((wim2, wim1, wi), 0)
                    p_tri = (count_tri + 1) / (count_bi_context + V)
                prob = lambdas[0] * p_uni + lambdas[1] * p_bi + lambdas[2] * p_tri

            if prob <= 0.0:
                tiny = 1e-300
                log2p = math.log2(tiny)
            else:
                log2p = math.log2(prob)

            total_log2 += log2p
            total_tokens += 1

            if verbose > 0:
                if n == 2:
                    print(f"P({wi}|{wim1}) = {lambdas[0]:.3f}*{p_uni:.6f} + {lambdas[1]:.3f}*{p_bi:.6f} = {prob:.6f}")
                else:
                    print(f"P({wi}|({wim2},{wim1})) = {lambdas[0]:.3f}*{p_uni:.6f} + {lambdas[1]:.3f}*{p_bi:.6f} + {lambdas[2]:.3f}*{p_tri:.6f} = {prob:.6f}")

    log_prob = total_log2  # log2 p(T) total (suma por token)
    H = -log_prob / total_tokens if total_tokens > 0 else float('inf')
    PP = 2 ** H
    return log_prob, H, PP, total_tokens

log_bi, H_bi, PP_bi, tk_bi = interpolation_eval(counts_blake, pseudo_lines, lambdas=[0.3, 0.7], n=2, verbose=0)
log_tri, H_tri, PP_tri, tk_tri = interpolation_eval(counts_blake, pseudo_lines, lambdas=[0.2, 0.3, 0.5], n=3, verbose=0)

table = [
    ["Log. Probabilidad", f"{log_bi:.4f}", f"{log_tri:.4f}"],
    ["H (bits/token)", f"{H_bi:.4f}", f"{H_tri:.4f}"],
    ["PP", f"{PP_bi:.4f}", f"{PP_tri:.4f}"]
]
print(tabulate(table, headers=["-", "BiInterpolado", "TriInterpolado"], tablefmt="fancy_grid"))

╒═══════════════════╤═════════════════╤══════════════════╕
│ -                 │   BiInterpolado │   TriInterpolado │
╞═══════════════════╪═════════════════╪══════════════════╡
│ Log. Probabilidad │      -4257.35   │       -4539.17   │
├───────────────────┼─────────────────┼──────────────────┤
│ H (bits/token)    │          9.2551 │           9.8678 │
├───────────────────┼─────────────────┼──────────────────┤
│ PP                │        611.034  │         934.308  │
╘═══════════════════╧═════════════════╧══════════════════╛


[nltk_data] Downloading package gutenberg to C:\Users\Shiyi Cheng
[nltk_data]     yi\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Shiyi Cheng
[nltk_data]     yi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Shiyi Cheng
[nltk_data]     yi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# EXTRA 1. Generación

Genera texto utilizando algunos de los modelos aprendidos. Se debe:

- empezar con el contexto inicial de frase.
- calcular la distribución de probabilidad para la siguiente palabra dado el contexto.
- elegir una palabra de esa distribución teniendo en cuenta su probabilidad.
- añadir la palabra a la secuencia y actualizar el contexto.
- repetir hasta generar un token </s>.

Haz una comparativa del resultado.


# EXTRA 2. Palabras desconocida

Hasta ahora el vocabulario de todas las frases de test estaba contenido en el vocabulario del modelo. Es posible que haya bigramas o trigramas no vistos en entrenamiento, por eso se suavizan los modelos, pero no unigramas. Esto es una situación ideal que pocas veces ocurre. Lo más habitual, excepto en tareas muy pequeñas, es que las muestras de test contengan palabras no vistas en entrenamiento y por tanto no contenidas en el vocabulario del modelo (out-of-vocabulary words). La manera más fácil de soluciónar esto es añadir al vocabulario una palabra ('<UNK>') que represente a todas las posibles palabras desconocidas con las que se encuentre el modelo. Esta palabra no se considerará para generar texto pero sí a efecto de asignarle una pequeña masa de probabilidad en el cálculo de la probabilidad de la frase.

Modifica las funciones necesarias para considerar la palabra '<UNK>' y utilizala cuando aparezca una palabra fuera del vocabulario del modelo.
    
Puedes hacer un análisis del rendimiento de los modelos utilizando el fichero 'pseudo_blake_out_of_vocabulary.txt'.

# EXTRA 3. Grid search

Escribe una función que busque la mejor combinación de valores de $\lambda$s en un modelo interpolado.

Hay dos cosas imprescindibles:

- necesitas un corpus de desarrollo distinto al de entrenamiento y al de test. Puede ser una parte del entrenamiento o del test pero en ese caso debes eliminar del entrenamiento o de la evaluar el resultado final.
- define una rejilla de valores para tus hiperparámetros (grid search) y prueba cada combinación y evaluala con el corpus de desarrollo.
- quedate con la combinación que mejor _PP_ obtenga.
- evalúa **una sola vez** esa combinación en el conjunto de test para obtener el resultado final.

Nota: si lo pruebas sobre el corpus de 'blake-poems.txt' deberás implementar antes el EXTRA 3.
